# MODEL INFERENCE SERVICE

In [ ]:
%pip install -U "bitsandbytes>=0.46.1"


In [ ]:
from pathlib import Path

import torch

from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MODEL_ID = "codellama/CodeLlama-7b-Instruct-hf"

ADAPTER_PATH = Path(
    "/content/drive/MyDrive/CodeMigrateAI/"
    "experiments/qlora_v2/"
    "codemigrate_qlora_v2_1_adapter"
)


# ------------------------------------------------------------
# 2. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# 3. Verify adapter
# ------------------------------------------------------------

assert (ADAPTER_PATH / "adapter_config.json").is_file()

assert (ADAPTER_PATH / "adapter_model.safetensors").is_file()

assert torch.cuda.is_available(), "GPU runtime required"

print("Adapter verified:", ADAPTER_PATH)
print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 4. Configure 4-bit quantization
# ------------------------------------------------------------

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)


# ------------------------------------------------------------
# 5. Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    str(ADAPTER_PATH)
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ------------------------------------------------------------
# 6. Load base CodeLlama
# ------------------------------------------------------------

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.bfloat16
)


# ------------------------------------------------------------
# 7. Attach fine-tuned LoRA weights
# ------------------------------------------------------------

model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_PATH),
    is_trainable=False
)

model.eval()

model.config.use_cache = True


# ------------------------------------------------------------
# 8. Model information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CODEMIGRATE AI - MODEL INFORMATION")
print("=" * 60)

print("Base model:", MODEL_ID)

print("Fine-tuned adapter:", ADAPTER_PATH.name)

print("Model device:", model.device)

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nFine-tuned model loaded successfully")

# CODE GENERATION

In [ ]:
import time

# ------------------------------------------------------------
# 1. Prompt builder
# ------------------------------------------------------------

def build_migration_prompt(
    instruction: str,
    legacy_code: str
) -> str:

    system_message = """
You are CodeMigrate AI, an expert C# migration assistant.

Migrate legacy .NET Framework code to modern .NET.

Requirements:
- Follow the migration instruction.
- Preserve business logic and observable behavior.
- Preserve class and method names unless explicitly instructed.
- Preserve filtering conditions and null behavior.
- Use valid modern .NET APIs.
- Include required namespaces.
- Return only the migrated C# code.
- Do not include explanations or Markdown.
""".strip()

    return (
        "<s>[INST] <<SYS>>\n"
        f"{system_message}\n"
        "<</SYS>>\n\n"
        f"Migration instruction:\n{instruction}\n\n"
        f"Legacy C# code:\n{legacy_code.strip()}\n"
        "[/INST]\n"
    )


# ------------------------------------------------------------
# 2. Migration generation
# ------------------------------------------------------------

def generate_migration(
    instruction: str,
    legacy_code: str,
    max_new_tokens: int = 384
) -> dict:

    prompt = build_migration_prompt(
        instruction,
        legacy_code
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    start_time = time.perf_counter()

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    generated_code = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    duration = time.perf_counter() - start_time

    return {
        "generated_code": generated_code,
        "generation_time": round(duration, 3),
        "generated_tokens": len(generated_ids)
    }


print("Migration generation function ready")

**Inference check**

In [ ]:
instruction = (
    "Migrate ConfigurationManager.AppSettings usage "
    "to modern .NET IConfiguration using constructor injection. "
    "Preserve the class name, method name, and fallback value."
)

legacy_code = """
using System.Configuration;

public class FileStorageOptions
{
    public string GetStorageFolder()
    {
        return ConfigurationManager.AppSettings["StorageFolder"]
            ?? "uploads";
    }
}
"""

result = generate_migration(
    instruction=instruction,
    legacy_code=legacy_code
)

print("=" * 60)
print("GENERATED C# CODE")
print("=" * 60)

print(result["generated_code"])

print("\nGeneration time:", result["generation_time"], "seconds")
print("Generated tokens:", result["generated_tokens"])

# Create the Model Inference API

In [ ]:
%pip install -q fastapi uvicorn httpx

In [ ]:
import secrets
import threading
import time
import torch

from typing import Literal

from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, Field


# ============================================================
# 1. API Configuration
# ============================================================

app = FastAPI(
    title="CodeMigrate AI Inference API",
    description="CodeLlama QLoRA Migration and CrewAI Agent Service",
    version="1.1.0"
)

API_KEY = secrets.token_urlsafe(32)

api_key_header = APIKeyHeader(
    name="X-API-Key",
    auto_error=False
)

# Prevent simultaneous requests from using the same model.
inference_lock = threading.Lock()


# ============================================================
# 2. Authentication
# ============================================================

def verify_api_key(
    provided_key: str | None = Depends(api_key_header)
):

    if (
        provided_key is None
        or not secrets.compare_digest(
            provided_key,
            API_KEY
        )
    ):
        raise HTTPException(
            status_code=401,
            detail="Invalid API key"
        )


# ============================================================
# 3. Request Schemas
# ============================================================

class MigrationRequest(BaseModel):

    instruction: str = Field(
        min_length=1,
        max_length=2000
    )

    legacy_code: str = Field(
        min_length=1,
        max_length=15000
    )


class ChatMessage(BaseModel):

    role: Literal[
        "system",
        "user",
        "assistant"
    ]

    content: str = Field(
        min_length=1,
        max_length=20000
    )


class ChatRequest(BaseModel):

    messages: list[ChatMessage] = Field(
        min_length=1,
        max_length=20
    )

    max_new_tokens: int = Field(
        default=256,
        ge=1,
        le=512
    )


# ============================================================
# 4. Health Endpoint
# ============================================================

@app.get("/health")
def health():

    return {
        "status": "ready",
        "base_model": MODEL_ID,
        "adapter": ADAPTER_PATH.name,
        "device": str(model.device),
        "endpoints": [
            "/health",
            "/migrate",
            "/chat"
        ]
    }


# ============================================================
# 5. Migration Endpoint — Fine-Tuned QLoRA
# ============================================================

@app.post(
    "/migrate",
    dependencies=[Depends(verify_api_key)]
)
def migrate(request: MigrationRequest):

    with inference_lock:

        result = generate_migration(
            instruction=request.instruction,
            legacy_code=request.legacy_code
        )

    return result


# ============================================================
# 6. Chat Endpoint — Base CodeLlama for CrewAI
# ============================================================

@app.post(
    "/chat",
    dependencies=[Depends(verify_api_key)]
)
def chat(request: ChatRequest):

    start_time = time.perf_counter()

    messages = [
        message.model_dump()
        for message in request.messages
    ]

    # CodeLlama expects a conversation ending with a user turn.
    if messages[-1]["role"] != "user":

        raise HTTPException(
            status_code=422,
            detail="The last chat message must have role 'user'."
        )

    with inference_lock:

        # -----------------------------------------------
        # Format conversation using CodeLlama tokenizer
        # -----------------------------------------------

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False
        ).to(model.device)

        input_length = inputs["input_ids"].shape[1]

        # -----------------------------------------------
        # Disable QLoRA adapter temporarily
        # -----------------------------------------------

        model.eval()

        with model.disable_adapter():

            with torch.inference_mode():

                outputs = model.generate(
                    **inputs,

                    max_new_tokens=request.max_new_tokens,

                    do_sample=False,

                    eos_token_id=tokenizer.eos_token_id,

                    pad_token_id=tokenizer.eos_token_id
                )

        # -----------------------------------------------
        # Decode newly generated tokens only
        # -----------------------------------------------

        generated_ids = outputs[0, input_length:]

        generated_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

    elapsed_time = time.perf_counter() - start_time

    return {
        "text": generated_text,
        "generated_tokens": len(generated_ids),
        "generation_time": round(elapsed_time, 3),
        "model": MODEL_ID,
        "adapter_used": False
    }


# ============================================================
# 7. Endpoint Verification
# ============================================================

print("=" * 60)
print("CODEMIGRATE AI INFERENCE API")
print("=" * 60)

print("Model:", MODEL_ID)
print("Fine-tuned adapter:", ADAPTER_PATH.name)

print("\nRegistered endpoints:")

for route in app.routes:

    if route.path in {
        "/health",
        "/migrate",
        "/chat"
    }:

        print(
            route.path,
            "|",
            getattr(route, "methods", None)
        )

print("\nAuthentication: X-API-Key")
print("API initialization: PASS")

# Test Local Inference Endpoints

Test `/health`, authentication, and `/migrate` against the current FastAPI
app before opening any public tunnel. The following chat test also verifies
that CodeLlama can respond with its QLoRA adapter temporarily disabled.


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)


# ------------------------------------------------------------
# Test 1: Health Check
# ------------------------------------------------------------

health_response = client.get("/health")

assert health_response.status_code == 200

print("--- API Health ---")
print(health_response.json())


# ------------------------------------------------------------
# Test 2: Authentication
# ------------------------------------------------------------

unauthorized_response = client.post(
    "/migrate",
    json={
        "instruction": "Migrate legacy C# code.",
        "legacy_code": "public class Test {}"
    }
)

assert unauthorized_response.status_code == 401

print("\nAuthentication: PASS")


# ------------------------------------------------------------
# Test 3: Migration
# ------------------------------------------------------------

migration_request = {
    "instruction": (
        "Migrate ConfigurationManager.AppSettings "
        "to modern .NET IConfiguration using "
        "constructor injection. Preserve the "
        "original method name and fallback value."
    ),

    "legacy_code": """
using System.Configuration;

public class FileStorageOptions
{
    public string GetStorageFolder()
    {
        return ConfigurationManager.AppSettings["StorageFolder"]
            ?? "uploads";
    }
}
"""
}


response = client.post(
    "/migrate",
    headers={
        "X-API-Key": API_KEY
    },
    json=migration_request
)

assert response.status_code == 200, response.text

result = response.json()

assert "generated_code" in result
assert "IConfiguration" in result["generated_code"]


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("API MIGRATION RESULT")
print("=" * 60)

print(result["generated_code"])

print("\nGeneration time:", result["generation_time"])
print("Generated tokens:", result["generated_tokens"])

print("\nAPI inference test: PASS")

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)


# ------------------------------------------------------------
# Test CrewAI Chat Endpoint
# ------------------------------------------------------------

response = client.post(
    "/chat",

    headers={
        "X-API-Key": API_KEY
    },

    json={
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a C# code analysis assistant. "
                    "Answer concisely."
                )
            },
            {
                "role": "user",
                "content": (
                    "What does this C# method do? "
                    "public bool IsActive(int id) "
                    "{ return id > 0; }"
                )
            }
        ],

        "max_new_tokens": 128
    }
)


print("HTTP Status:", response.status_code)

if response.status_code == 200:

    result = response.json()

    print("\nModel:", result["model"])
    print("Adapter used:", result["adapter_used"])

    print("\nGenerated response:")
    print(result["text"])

    print("\nGenerated tokens:", result["generated_tokens"])

else:

    print("Error:", response.text)


assert response.status_code == 200

print("\nCrewAI Chat Endpoint: PASS")

# Start or Restart the FastAPI Server

Run this cell **after** defining all three endpoints. On reruns, it stops
the previous Uvicorn server managed by this notebook and starts the current
`app` object. A successful `/health` response alone is not enough: the
actual running server must expose `/health`, `/migrate`, and `/chat`.


In [ ]:
import threading
import time
import requests
import uvicorn


API_PORT = 8000
LOCAL_URL = f"http://127.0.0.1:{API_PORT}"
REQUIRED_ENDPOINTS = {"/health", "/migrate", "/chat"}

declared_endpoints = {route.path for route in app.routes}
missing = REQUIRED_ENDPOINTS - declared_endpoints
if missing:
    raise RuntimeError(
        f"Current FastAPI app is missing endpoints: {sorted(missing)}. "
        "Run the API Configuration cell first."
    )


# Always stop the old server, even when GET /health returns 200.
# The old instance might reference a different FastAPI app object.
if "api_server" in globals() and isinstance(api_server, uvicorn.Server):
    api_server.should_exit = True

    previous_thread = globals().get("server_thread")
    if previous_thread is not None and previous_thread.is_alive():
        previous_thread.join(timeout=15)

        if previous_thread.is_alive():
            raise RuntimeError("The previous Uvicorn server did not stop.")


# A listener without a notebook-managed server handle must be stopped
# separately; silently reusing it would repeat the stale /chat 404 bug.
try:
    existing = requests.get(f"{LOCAL_URL}/openapi.json", timeout=2)
except requests.RequestException:
    existing = None

if existing is not None:
    raise RuntimeError(
        f"Port {API_PORT} is still occupied (HTTP {existing.status_code}). "
        "Stop that server before starting this notebook's Uvicorn cell."
    )


config = uvicorn.Config(
    app=app,
    host="127.0.0.1",
    port=API_PORT,
    log_level="warning",
)
api_server = uvicorn.Server(config)
server_thread = threading.Thread(target=api_server.run, daemon=True)
server_thread.start()


for attempt in range(30):
    try:
        response = requests.get(f"{LOCAL_URL}/openapi.json", timeout=2)
        if response.ok:
            served_endpoints = set(response.json().get("paths", {}))
            if REQUIRED_ENDPOINTS <= served_endpoints:
                break

            raise RuntimeError(
                f"The live server is missing: "
                f"{sorted(REQUIRED_ENDPOINTS - served_endpoints)}"
            )
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("FastAPI did not start within 30 seconds.")


health_response = requests.get(f"{LOCAL_URL}/health", timeout=5)
health_response.raise_for_status()

print("=" * 60)
print("CODEMIGRATE AI - LIVE FASTAPI SERVER")
print("=" * 60)
print("Server:", LOCAL_URL)
print("Status:", health_response.json()["status"])
print("Model:", health_response.json()["base_model"])
print("Live endpoints:", sorted(served_endpoints & REQUIRED_ENDPOINTS))
print("FastAPI server: PASS")


# Create a Temporary HTTPS Tunnel

The Cloudflare tunnel forwards public requests to `LOCAL_URL`.
Re-running the tunnel cell replaces the old public URL. Keep the notebook
and Colab runtime active while the remote Spyder client needs the model.


Install `cloudflared` if needed, then start the tunnel and check that
the **public** FastAPI application exposes all three endpoints.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared


In [ ]:
import queue
import re
import subprocess
import threading
import time
import requests


# Close the previous tunnel before requesting a new public address.
previous_tunnel = globals().get("tunnel_process")
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_tunnel.kill()
        previous_tunnel.wait(timeout=5)


tunnel_process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        LOCAL_URL,
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url_queue = queue.Queue()


def capture_tunnel_url():
    for line in tunnel_process.stdout:
        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line,
        )
        if match:
            url_queue.put(match.group(0))
            break


reader_thread = threading.Thread(
    target=capture_tunnel_url,
    daemon=True,
)
reader_thread.start()

try:
    PUBLIC_URL = url_queue.get(timeout=60)
except queue.Empty as error:
    raise RuntimeError("Cloudflare tunnel URL was not generated.") from error


# Verify the public route list, not just the in-process FastAPI app.
for attempt in range(20):
    try:
        response = requests.get(
            f"{PUBLIC_URL}/openapi.json", timeout=6
        )
        if response.ok:
            public_endpoints = set(response.json().get("paths", {}))
            if REQUIRED_ENDPOINTS <= public_endpoints:
                break
    except (requests.RequestException, ValueError):
        pass
    time.sleep(2)
else:
    raise RuntimeError(
        "The Cloudflare URL does not expose /health, /migrate and /chat. "
        "Check that the tunnel forwards to the current LOCAL_URL."
    )


print("=" * 60)
print("CODEMIGRATE AI - REMOTE INFERENCE")
print("=" * 60)
print("Public API URL:", PUBLIC_URL)
print("Migration endpoint:", f"{PUBLIC_URL}/migrate")
print("Agent endpoint:", f"{PUBLIC_URL}/chat")
print("Public endpoints:", sorted(public_endpoints & REQUIRED_ENDPOINTS))
print("Authentication: X-API-Key")
print("Tunnel initialized: PASS")


# Verify Remote Access and Copy Credentials

The unauthenticated `/chat` request must return HTTP 401. That proves the
route exists and its API-key protection is active, without running the
model. Copy the current public URL to the Spyder client. The runtime API
key is sensitive; reveal it only in your private Colab session if needed.


In [ ]:
import requests

public_schema = requests.get(f"{PUBLIC_URL}/openapi.json", timeout=15)
public_schema.raise_for_status()
assert REQUIRED_ENDPOINTS <= set(public_schema.json()["paths"])

unauthorized_chat = requests.post(
    f"{PUBLIC_URL}/chat",
    json={
        "messages": [{"role": "user", "content": "Hello"}],
        "max_new_tokens": 8,
    },
    timeout=15,
)
assert unauthorized_chat.status_code == 401, (
    unauthorized_chat.status_code,
    unauthorized_chat.text[:300],
)

print("Public /chat route: PASS (HTTP 401 without a key)")
print("Use PUBLIC_URL in your Spyder agent client:", PUBLIC_URL)
print("The current API key is in the API_KEY variable.")
print("If you need to copy it privately, run: print(API_KEY)")
print("Do not share or commit the printed key.")


In [ ]:
print(API_KEY)